# Functional tests

Below is a list of functional tests which should be included in sending and receiving implementations.

### Sending

- Ensure taproot outputs are excluded during coin selection if the sender does not have access to the key path private key (unless using H as the taproot internal key)
- Ensure the silent payment address is re-derived if inputs are added or removed during RBF

### Receiving

- Ensure the public key can be extracted from non-standard P2PKH scriptSigs
- Ensure taproot script path spends are included, using the taproot output key (unless H is used as the taproot internal key)
- Ensure the scanner can extract the public key from each of the input types supported (e.g. P2WPKH, P2SH-P2WPKH, etc.)

In [ ]:
import json
import os
import sys
from collections import Counter
import copy
from typing import Tuple, List, Dict


sys.path.append(os.path.abspath('..'))

from send import sending_run
from receive import receiving_run

# Percorso del file JSON (modificalo se necessario)
TEST_FILE = './test_vectors.json' 

def run_all_sending_tests() -> bool:
    try:
        with open(TEST_FILE, 'r') as f:
            test_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Errore: File {TEST_FILE} non trovato. Controlla il percorso!")
        return

    passed = 0
    failed = 0
    failed_tests = []

    print(f"\n{'='*70}")
    print(f"  Silent Payments — Sending test suite ({len(test_data)} vettori)")
    print(f"{'='*70}\n")

    for i, test in enumerate(test_data):
        comment = test.get('comment', f'Test #{i}')
        
        # Salta i test che non hanno la sezione 'sending'
        if 'sending' not in test or not test['sending']:
            print(f"  [{i:>2}] {comment} \n   -> ⏩ SALTATO (Nessun vettore sending)")
            continue
            
        sending_case = test['sending'][0]
        given = sending_case['given']
        expected = sending_case['expected']

        vin = given['vin']
        recipients = given['recipients']
        
        try:
            # Eseguiamo la funzione di invio
            result = sending_run(vin, recipients)
            
            # Estraiamo TUTTI i possibili set di output validi
            expected_outputs_sets = expected.get('outputs', [])
            if not expected_outputs_sets:
                expected_outputs_sets = [[]] 
            
            test_passed = False
            # Iteriamo su tutte le combinazioni valide fornite dal JSON
            for expected_outputs in expected_outputs_sets:
                if Counter(result) == Counter(expected_outputs):
                    test_passed = True
                    break
            
            if test_passed:
                print(f"  [{i:>2}] {comment} ✓ PASS")
                passed += 1
            else:
                print(f"  [{i:>2}] {comment} ✗ FAIL")
                print(f"   Atteso (una di queste combinazioni): {expected_outputs_sets}")
                print(f"   Ottenuto: {result}")
                failed += 1
                failed_tests.append(i)
                
        except Exception as e:
            expected_outputs_sets = expected.get('outputs', [])
            # Se il test si aspettava un fallimento (array vuoto o assente)
            if not expected_outputs_sets or expected_outputs_sets == [[]]:
                print(f"  [{i:>2}] {comment} ✓ PASS (Eccezione catturata: {e})")
                passed += 1
            else:
                print(f"  [{i:>2}] {comment} ⚠️ ERRORE INATTESO")
                print(f"   Eccezione Python: {e}")
                failed += 1
                failed_tests.append(i)

    print(f"\n{'='*70}")
    print(f"  Risultato: {passed}/{passed + failed} passati", end='')
    if failed > 0:
        print(f"  —  falliti: {failed_tests}")
        ret = False
    else:
        print("  🎉")
        ret = True
    print(f"{'='*70}\n")
    return ret

def deep_clone(x):
    return copy.deepcopy(x)

def normalize_outputs(outputs: List[Dict]) -> List[tuple]:
    """Ordina gli output del wallet per garantire un confronto affidabile."""
    return sorted(
        (o['pub_key'], o['priv_key_tweak'])
        for o in outputs
    )

def run_all_receiving_tests() -> bool:
    try:
        with open(TEST_FILE, 'r') as f:
            vectors = json.load(f)
    except FileNotFoundError:
        print(f"❌ Errore: File {TEST_FILE} non trovato. Controlla il percorso!")
        return

    total = len(vectors)
    passed = 0
    failed = []

    print(f"\n{'='*70}")
    print(f"  Silent Payments — Receiving test suite ({total} vettori)")
    print(f"{'='*70}\n")

    for i, raw_data in enumerate(vectors):
        data = deep_clone(raw_data)
        comment = data.get('comment', f'test #{i}')
        receiving_cases = data.get('receiving', [])

        if not receiving_cases:
            print(f"  [{i:>2}] {comment} \n   -> ⏩ SALTATO (Nessun caso receiving)")
            continue

        all_cases_passed = True
        details = []

        for case_idx, receiving in enumerate(receiving_cases):
            given = receiving.get('given', {})
            expected = receiving.get('expected', {})

            vin = deep_clone(given.get('vin'))
            outputs = deep_clone(given.get('outputs', []))
            key_material = deep_clone(given.get('key_material'))
            labels = deep_clone(given.get('labels'))

            expected_addresses = expected.get('addresses', [])
            expected_outputs = expected.get('outputs', [])

            try:
                addresses, wallet = receiving_run(
                    vin=vin,
                    outputs=outputs,
                    key_material=key_material,
                    labels=labels if labels else None,
                )
            except Exception as e:
                all_cases_passed = False
                details.append(f'   -> Eccezione inattesa: {e}')
                continue

            # Confronti
            addr_ok = sorted(addresses) == sorted(expected_addresses)
            out_ok = normalize_outputs(wallet) == normalize_outputs(expected_outputs)

            if not (addr_ok and out_ok):
                all_cases_passed = False
                if not addr_ok:
                    details.append(f'   -> Errore Address: Atteso {expected_addresses}, Ottenuto {addresses}')
                if not out_ok:
                    details.append(f'   -> Errore Outputs: Atteso {expected_outputs}, Ottenuto {wallet}')

        if all_cases_passed:
            print(f"  [{i:>2}] {comment} ✓ PASS")
            passed += 1
        else:
            print(f"  [{i:>2}] {comment} ✗ FAIL")
            for d in details:
                print(d)
            failed.append(i)

    print(f"\n{'='*70}")
    print(f"  Risultato: {passed}/{total} passati", end='')
    if failed:
        print(f"  —  falliti: {failed}")
        ret = False
    else:
        print("  🎉")
        ret = True
    print(f"{'='*70}\n")
    return ret

if __name__ == "__main__":
    print("🚀 Avvio della test suite per Silent Payments...\n")
    send_passed = False
    receive_passed = False
    try:
        print("🔍 Esecuzione dei test di INVIO (sending)...")
        send_passed = run_all_sending_tests()
    except BaseException as e:
        print(f"\n🚨 CRASH CRITICO DURANTE I TEST SENDING: {e}")
        
    print("\n" + "X"*70 + "\n") # Linea di separazione gigante per farsi notare
    
    try:
        print("🔍 Esecuzione dei test di RICEZIONE (receiving)...")
        receive_passed = run_all_receiving_tests()
    except BaseException as e:
        print(f"\n🚨 CRASH CRITICO DURANTE I TEST RECEIVING: {e}")
        
    if send_passed and receive_passed:
        print("\nTUTTI I TEST SONO PASSATI!")
    print("\n✅ Fine dell'esecuzione del Notebook.")